In [1]:
from google.colab import drive

# Mount Google Drive at /content/drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/BERT_SMM4H

/content/drive/MyDrive/BERT_SMM4H


In [3]:
import torch
import pandas as pd
import re
# import string
# from nltk.corpus import stopwords
# from nltk.stem import WordNetLemmatizer
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
#import evaluate
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

In [4]:
BATCH_SIZE = 8
NUM_PROCS = 32
LR = 7.21422e-06
WEIGHT_DECAY = 0.00694763
EPOCHS = 3

# Load the training data
train_data = pd.read_csv('SMM4H-2024-Task5-Training.tsv', sep='\t')
# Remove the "twitter_id" column
train_data = train_data.drop(columns=["tweet_id"])

# Load the validation data
val_data = pd.read_csv('SMM4H-2024-Task5-Validation.tsv', sep='\t')
# Remove the "twitter_id" column
val_data = val_data.drop(columns=["tweet_id"])

# View the first few rows of the training data
print(train_data.head())
# View the first few rows of the validation data
print(val_data.head())

                                                text  label
0  Psych Med Twitter, wondering if you can help. ...      0
1  @dmx_biographer @HellaChillAF I wouldn’t fuck ...      0
2  Harley is autistic.... she has the mental capa...      1
3  I just made my son cry before going to school ...      1
4  Yooooooo my dad really just said “I’d rather h...      0
                                                text  label
0  Mom Finds Secret to Homeschooling her Autistic...      0
1  I hope our child has my calmness and NOT Andre...      0
2  Blind Girl With Autism Mimics Whitney Houston ...      0
3  @amyschumer watching #growing, my 2yo son has ...      1
4  @robbystarbuck @AmericanAir would rather a mot...      0


In [5]:
import html
# Define preprocessing function
def preprocess_tweet(tweet):
    tweet = tweet.lower()
    # Remove URLs
    cleaned_tweet = re.sub(r'http\S+|www\S+|https\S+', '', tweet)

    # Remove mentions and hashtags
    cleaned_tweet = re.sub(r'@\w+', '', cleaned_tweet)
    cleaned_tweet = re.sub(r'#', '', cleaned_tweet)

    # Remove unscape HTML tags
    cleaned_tweet = html.unescape(cleaned_tweet)

    # Remove extra spaces, tabs, and newlines
    cleaned_tweet = re.sub(r'\s+', ' ', cleaned_tweet).strip()

    return cleaned_tweet


train_data["text"] = train_data["text"].apply(preprocess_tweet)
val_data["text"] = val_data["text"].apply(preprocess_tweet)

# View the first few rows of the training data
print(train_data.head())
# View the first few rows of the validation data
print(val_data.head())

                                                text  label
0  psych med twitter, wondering if you can help. ...      0
1  i wouldn’t fuck rob with your dick. he’s an au...      0
2  harley is autistic.... she has the mental capa...      1
3  i just made my son cry before going to school ...      1
4  yooooooo my dad really just said “i’d rather h...      0
                                                text  label
0  mom finds secret to homeschooling her autistic...      0
1  i hope our child has my calmness and not andre...      0
2  blind girl with autism mimics whitney houston ...      0
3  watching growing, my 2yo son has asd, you just...      1
4  would rather a mother suffocate an asthmatic 2...      0


In [6]:
id2label = {
    0: "Negative",
    1: "Positive",
}
label2id = {
    "Negative": 0,
    "Positive": 1,
}


# Convert train_data and val_data from Pandas DataFrames into Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_data)
valid_dataset = Dataset.from_pandas(val_data)

In [7]:
# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Helper function for preprocessing.
def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
    )

#tokenizing training data
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=BATCH_SIZE,
    num_proc=NUM_PROCS
)
#tokenizing validation data
tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=BATCH_SIZE,
    num_proc=NUM_PROCS
)

# Initialize data collator.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map (num_proc=32):   0%|          | 0/7398 [00:00<?, ? examples/s]

Map (num_proc=32):   0%|          | 0/389 [00:00<?, ? examples/s]

In [8]:
!pip install evaluate

In [9]:
import evaluate

accuracy = evaluate.load('accuracy')
f1_score = evaluate.load('f1')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    #calculating the accuracy
    #accuracy_result = accuracy.compute(predictions=predictions, references=labels)
    #calulating the F1 score
    f1_score_result = f1_score.compute(predictions=predictions, references=labels)
    return f1_score_result

In [10]:
# This cell is basically for loading the model

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
# Define the training arguments
training_args = TrainingArguments(
    output_dir="smm4h_bert_base_clean_v3",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    # logging_dir="./logs",
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="tensorboard",
    fp16=torch.cuda.is_available(),
)


# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    callbacks=[],
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

history = trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.471292,0.355101,0.776471
2,0.312421,0.322767,0.819923
3,0.238161,0.350736,0.837037


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

In [13]:
# Get predictions for the evaluation dataset using the best model
eval_predictions = trainer.predict(tokenized_valid)

# Access the predicted labels from the output
predicted_labels = eval_predictions.predictions.argmax(axis=1)  # Get class indices

# Access the predicted probabilities for class "1" (or any other class) from the output
predicted_probabilities_class_1 = eval_predictions.predictions[:, 1]  # Assuming class "1" is index 1

# You can now process or analyze the predicted labels for the evaluation set
print(f"Predicted labels for evaluation data: {predicted_labels}")

# You can now process or analyze the predicted probabilities for the evaluation set
print(f"Predicted probabilities for class 1 (logit values): {predicted_probabilities_class_1}")


# Apply softmax to convert logits to probabilities
probabilities = np.exp(eval_predictions.predictions) / np.sum(np.exp(eval_predictions.predictions), axis=1, keepdims=True)

# Now you can access the normalized probabilities for class "1"
predicted_probabilities_class_1_normalized = probabilities[:, 1]

# You can now process or analyze the predicted probabilities for the evaluation set
print(f"Predicted probabilities for class 1 (normalized values): {predicted_probabilities_class_1_normalized}")

Predicted labels for evaluation data: [1 0 0 1 0 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0 0 1 0 0 0
 0 1 0 1 0 0 0 0 0 1 1 1 1 0 0 0 0 1 1 1 0 1 1 0 1 0 1 1 1 0 0 0 1 0 0 0 1
 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 0 0 1 1 0 1 1 0 0 0 0 1 0 1 0 1 0 0 0 1 1 0
 0 1 1 0 0 1 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 1 0 0 0 1 0
 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 1 1 1 0 1 1 0 0 0 0 0
 1 0 1 0 1 1 0 0 0 0 0 0 1 0 1 1 0 0 1 0 0 1 1 1 0 1 0 1 1 0 0 1 0 0 0 1 0
 1 1 0 1 0 0 1 1 0 1 0 0 0 0 0 0 0 1 1 1 0 0 1 0 0 1 0 1 0 0 0 0 0 1 0 0 1
 1 0 1 0 0 1 1 0 1 1 0 1 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0 0 1 1 1 0 0 1
 0 0 1 0 0 0 0 0 1 0 1 0 0 1 0 1 0 1 1 1 0 1 0 0 0 0 0 0 0 0 0 0 1 0 1 0 1
 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 1 1 1 0 1 1 0 0 1 0 0 1 1 1 1 0 0 0
 1 1 1 0 0 0 0 0 0 0 0 0 1 0 1 0 0 1 1]
Predicted probabilities for class 1 (logit values): [ 0.21008301 -2.328125   -2.2617188   1.5917969  -2.3105469  -2.4101562
 -1.4658203  -2.4355469  -2.203125   -2.3398438 